<a href="https://colab.research.google.com/github/ponkpook/nlp_practical_2025_sEXism/blob/feat%2Fnlp-task1.1-1.2-1.3/nlp_total_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install datasets scikit-learn pandas torch torchvision torchaudio numpy
# !pip uninstall -y transformers accelerate peft numpy
!pip install "transformers==4.39.3" "accelerate==0.30.0" "peft==0.10.0" "numpy==1.26.0"


  Using cached accelerate-0.30.0-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.10.0-py3-none-any.whl.metadata (13 kB)
Using cached accelerate-0.30.0-py3-none-any.whl (302 kB)
Using cached peft-0.10.0-py3-none-any.whl (199 kB)


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import ast
import accelerate, transformers
from google.colab import drive
from sklearn.metrics import accuracy_score, f1_score, classification_report
drive.mount('/content/drive')
# 文件路径就是 /content/drive/MyDrive/你的文件夹/文件名
print("accelerate:", accelerate.__version__)
print("transformers:", transformers.__version__)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
accelerate: 0.30.0
transformers: 4.39.3


In [7]:
# ===============================================================
# 0. 依赖
# pip install -U numpy torch transformers datasets scikit-learn
# ===============================================================
import ast, json, numpy as np, pandas as pd, torch, torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from transformers import EarlyStoppingCallback
from transformers import (AutoTokenizer, Trainer, TrainingArguments,
                          DataCollatorWithPadding, AutoModel)
from datasets import Dataset
# -----------------------------
CSV_PATH = "/content/EXIST2025_cleaned_plus_AEDA_fullinfo.csv"
MODEL_NAME = "vinai/bertweet-base"
MAX_LEN = 128
W1, W2, W3 = 0.7, 1.3, 2.0
# ===============================================================
# 1. 读 csv 并构造软标签
# ---------------------------------------------------------------
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/EXIST2025_cleaned_plus_AEDA_fullinfo.csv')

# ---- 1.1 只要 en / es 并且 annotator 标注存在 ----
df = df[df["lang"].isin(["en", "es"]) & df["labels_task1_1"].notnull()].copy()

def make_soft_task1(row):
    # >3 YES 归为 [0,1]；否则 [1,0]
    labels = ast.literal_eval(row["labels_task1_1"])
    yes = labels.count("YES")
    no  = labels.count("NO")
    return [0., 1.] if yes > no else [1., 0.]

def make_soft_task2(row):
    # 只对 task1 被判 YES 的 annotator 才算，"-" 忽略
    if make_soft_task1(row)[1] == 0:      # task1=>NO
        return [0.,0.,0.,1.]              # NO 类独占概率 1
    ints = [l for l in ast.literal_eval(row["labels_task1_2"])
            if l in ["DIRECT","REPORTED","JUDGEMENTAL"]]
    if not ints:
        return [0.,0.,0.,1.]
    p = [ints.count("DIRECT"),
         ints.count("REPORTED"),
         ints.count("JUDGEMENTAL")]
    p = np.asarray(p, dtype=np.float32)
    p = p / p.sum()
    return p.tolist() + [0.]              # add dummy “NO”=0 概率

T3_CATS = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE",
]                                   # len = 5,  最后会再加一个 “NO”

def make_soft_task3(row):
    # 如果 task1 判定为 NOT-sexist → [0,0,0,0,0,1]
    if make_soft_task1(row)[1] == 0:
        return [0., 0., 0., 0., 0., 1.]

    # 统计 6 位 annotator 各自选了哪些子类
    ann_lists = ast.literal_eval(row["labels_task1_3"])  # -> list[list[str]]
    counter = np.zeros(len(T3_CATS), dtype=np.float32)

    for ann in ann_lists:
        for c in ann:
            if c in T3_CATS:
                counter[T3_CATS.index(c)] += 1

    if counter.sum() == 0:                          # 全部是 "-" 或 "UNKNOWN"
        return [0., 0., 0., 0., 0., 1.]

    probs = counter / counter.sum()                 # 归一化
    return probs.tolist() + [0.]                    # 末尾再加 1 个 “NO” 概率 0

df["soft_label_task3"] = df.apply(make_soft_task3, axis=1)

df["soft_label_task1"] = df.apply(make_soft_task1, axis=1)
df["soft_label_task2"] = df.apply(make_soft_task2, axis=1)

# ===============================================================
# 2. 计算 annotator one-hot→dense 向量
# ---------------------------------------------------------------
gender_vals = ["F", "M"]
age_vals    = ["18-22", "23-45", "46+"]
eth_vals    = ["White or Caucasian","Black or African American","Hispano or Latino",
               "Asian","Middle Eastern","Multiracial","other"]
g2i = {v:i for i,v in enumerate(gender_vals)}
a2i = {v:i for i,v in enumerate(age_vals)}
e2i = {v:i for i,v in enumerate(eth_vals)}

def build_annot_vec(ex):
    g = ast.literal_eval(ex["gender_annotators"])
    a = ast.literal_eval(ex["age_annotators"])
    e = ast.literal_eval(ex["ethnicities_annotators"])
    v = np.zeros(len(gender_vals)+len(age_vals)+len(eth_vals), dtype=np.float32)
    for gg,aa,ee in zip(g,a,e):
        v[g2i.get(gg,0)] += 1
        v[len(gender_vals)+a2i.get(aa,0)] += 1
        v[len(gender_vals)+len(age_vals)+e2i.get(ee,"other")] += 1
    v = v / len(g)          # 6 annotators →平均
    return v.tolist()

df["annot_vector"] = df.apply(build_annot_vec, axis=1)

# ===============================================================
# 3. HuggingFace Dataset + tokenizer
# ---------------------------------------------------------------
tok = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

def tokenize(batch):
    return tok(batch["text"], padding="max_length", truncation=True,
               max_length=MAX_LEN)

ds = Dataset.from_pandas(df[["text","soft_label_task1","soft_label_task2","soft_label_task3", "annot_vector"]])
ds = ds.map(tokenize, batched=True, remove_columns=["text"])

train_ds, val_ds = ds.train_test_split(test_size=0.1, seed=42).values()

# set_format 只保留需要 tensor 化的列
cols = [
    "input_ids", "attention_mask",
    "soft_label_task1", "soft_label_task2", "soft_label_task3",
    "annot_vector"
]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch",   columns=cols)
print(train_ds.column_names)
print(train_ds[0].keys())          # 第 0 条样本应包含 soft_label_task1

missing = [i for i,x in enumerate(train_ds) if "soft_label_task1" not in x]
print("num missing =", len(missing))   # 应该是 0

# ===============================================================
# 4. DataCollator —— 把 list → tensor，并喂给模型
# ---------------------------------------------------------------
class Collator:
    def __init__(self):
        self.base = DataCollatorWithPadding(tok, return_tensors="pt")

    def __call__(self, features):
        batch = self.base([{k: v for k, v in f.items()
                            if k in ["input_ids", "attention_mask"]}
                           for f in features])

        def stack(key):                          # 小工具
            return torch.stack([torch.as_tensor(f[key], dtype=torch.float32)
                                for f in features])

        batch["soft_label_task1"] = stack("soft_label_task1")
        batch["soft_label_task2"] = stack("soft_label_task2")
        batch["soft_label_task3"] = stack("soft_label_task3")   # ← 新增
        batch["annot_vector"]     = stack("annot_vector")
        return batch

collator = Collator()

# ===============================================================
# 5. 多任务模型
# ---------------------------------------------------------------
class MultiTaskModel(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size
        self.annot_fc = nn.Linear(len(gender_vals)+len(age_vals)+len(eth_vals), 16)
        self.class1   = nn.Linear(h+16, 2)
        self.class2   = nn.Linear(h+16, 4)
        self.class3 = nn.Linear(h+16, 6)  # num_labels_task3 = 5
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        annot_vector=None,
        soft_label_task1=None,   # ← 新增
        soft_label_task2=None,   # ← 新增
        soft_label_task3=None,   # ← 新增
    ):  # **kwargs 防止意外键
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:,0]        # [B,h]
        if annot_vector is not None:
            ann = self.annot_fc(annot_vector)
            rep = torch.cat([cls, ann], dim=1)
        else:
            rep = cls
        logits_task1 = self.class1(rep)
        logits_task2 = self.class2(rep)
        logits_task3 = self.class3(rep)          # 新增 Task 1.3 的输出
        return {
            'logits_task1': logits_task1,
            'logits_task2': logits_task2,
            'logits_task3': logits_task3
        }

model = MultiTaskModel(MODEL_NAME)

def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def compute_metrics(eval_pred):
    """
    eval_pred 由 Trainer 传入，通常是 (preds, labels)：
      ▸ preds  可以是 tuple / dict / ndarray（取决于 model.forward 返回）
      ▸ labels 从数据集里来，这里依旧用 val_ds 里的软标签做 “真值”
    """
    logits, _ = eval_pred        # labels 不直接用，真值从 val_ds 里拿软标签→硬标签

    # -------- 1. 拆分三套 logits --------
    logits1 = logits2 = logits3 = None

    if isinstance(logits, tuple):           # forward 返回 tuple
        if len(logits) == 3:                # (logits1, logits2, logits3)
            logits1, logits2, logits3 = logits
        elif len(logits) == 2:              # (logits1, logits2)
            logits1, logits2 = logits
    elif isinstance(logits, dict):          # forward 返回 dict
        logits1 = logits["logits_task1"]
        logits2 = logits.get("logits_task2")
        logits3 = logits.get("logits_task3")
    else:                                   # 单一输出
        logits1 = logits                    # 只 Task1

    metrics = {}

    # -------- 2. Task-1（二分类） --------
    if logits1 is not None:
        y_pred1 = np.argmax(logits1, axis=-1)
        y_true1 = np.argmax(val_ds["soft_label_task1"], axis=-1)      # 硬标签
        metrics.update({
            "t1_acc"        : accuracy_score(y_true1, y_pred1),
            "t1_f1_macro"   : f1_score(y_true1, y_pred1, average="macro"),
            "t1_f1_weighted": f1_score(y_true1, y_pred1, average="weighted"),
        })

    # -------- 3. Task-2（四分类） --------
    if logits2 is not None:
        y_pred2 = np.argmax(logits2, axis=-1)
        y_true2 = np.argmax(val_ds["soft_label_task2"], axis=-1)
        metrics.update({
            "t2_acc"        : accuracy_score(y_true2, y_pred2),
            "t2_f1_macro"   : f1_score(y_true2, y_pred2, average="macro"),
            "t2_f1_weighted": f1_score(y_true2, y_pred2, average="weighted"),
        })

    # -------- 4. Task-3（五标签多标签） --------
    if logits3 is not None:
        # 1) 把 logits 过 sigmoid → 概率
        proba3   = _sigmoid(logits3)

        # 2) 设阈值（0.5 最常见，也可以用验证集调优）
        y_pred3  = (proba3 >= 0.5).astype(int)      # shape (B, 5)

        # 3) 软标签 → 硬标签（>=0.5 视为被选中）
        y_true3 = (val_ds["soft_label_task3"].numpy() >= 0.5).astype(int)

        # 4) 多标签 F1：
        metrics.update({
            "t3_acc"        : accuracy_score(y_true3, y_pred3),
            "t3_f1_micro" : f1_score(y_true3, y_pred3, average="micro", zero_division=0),
            "t3_f1_macro" : f1_score(y_true3, y_pred3, average="macro", zero_division=0),
            "t3_f1_weight": f1_score(y_true3, y_pred3, average="weighted", zero_division=0),
        })

    return metrics

# ===============================================================
# 6. 自定义 Trainer 只写 compute_loss
# ---------------------------------------------------------------
def kl(logits, target, T=3):
    return F.kl_div(F.log_softmax(logits/T,-1), target, reduction="batchmean")*(T**2)
class MTTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        # ------- 1. 前向传播 -------
        outputs = model(
            input_ids      = inputs["input_ids"],
            attention_mask = inputs["attention_mask"],
            annot_vector   = inputs["annot_vector"],
        )                                # 得到 3 个 logits

        # ------- 2. 取目标（软）标签 -------
        tgt1 = inputs["soft_label_task1"]      # shape: (B, 2)
        tgt2 = inputs["soft_label_task2"]      # shape: (B, 4)
        tgt3 = inputs["soft_label_task3"]      # shape: (B, 5)  — 多标签

        # ------- 3. 分别计算每个任务的损失 -------
        # Task 1 / Task 2 用 KL 散度（软标签 → softmax）
        loss1 = F.kl_div(
            F.log_softmax(outputs["logits_task1"], dim=-1), tgt1,
            reduction="batchmean"
        )
        loss2 = F.kl_div(
            F.log_softmax(outputs["logits_task2"], dim=-1), tgt2,
            reduction="batchmean"
        )

        # Task 3 是多标签 → 用 BCE-with-logits
        loss3 = F.binary_cross_entropy_with_logits(
            outputs["logits_task3"], tgt3
        )

        # ------- 4. 可选：若 tweet 判定为非性别歧视 (Task1=NO)，不计入 task2/3 损失 -------
        # mask_yes = tgt1[:, 1] > 0           # “YES” 这一列的概率 > 0 视为正样本
        # loss2 = (loss2 * mask_yes).mean()
        # loss3 = (loss3 * mask_yes).mean()

        # ------- 5. 按权重加权 -------
        total_loss = W1 * loss1 + W2 * loss2 + W3 * loss3  # 例：自己设权重
        # 也可写成 total_loss = a*loss1 + b*loss2 + c*loss3

        return (total_loss, outputs) if return_outputs else total_loss

df['hard_t2'] = df['soft_label_task2'].apply(lambda x: np.argmax(x))
print(df['hard_t2'].value_counts())

# ===============================================================
# 7. 训练
# ---------------------------------------------------------------
args = TrainingArguments(
    output_dir="./multitask_soft",
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    num_train_epochs=30,
    learning_rate=2e-5,
    fp16=True,
    evaluation_strategy="epoch",   # 保存和评估都按 epoch
    save_strategy="epoch",         # 新增
    logging_strategy="epoch",
    report_to=[],
    metric_for_best_model="eval_loss",
    load_best_model_at_end=True,
    greater_is_better=False,
    seed=42,
)

trainer = MTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,  # 这里才是
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

trainer.train()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

['soft_label_task1', 'soft_label_task2', 'soft_label_task3', 'annot_vector', 'input_ids', 'token_type_ids', 'attention_mask']
dict_keys(['soft_label_task1', 'soft_label_task2', 'soft_label_task3', 'annot_vector', 'input_ids', 'attention_mask'])
num missing = 0


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

hard_t2
3    6346
0    2314
1     925
2     595
Name: count, dtype: int64


ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.21.0`: Please run `pip install transformers[torch]` or `pip install accelerate -U`

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import numpy as np

# 用 Trainer 的 predict 方法获得 logits
eval_result = trainer.predict(val_ds)
logits_task1 = eval_result.predictions[0]  # [N, 2]
logits_task2 = eval_result.predictions[1]  # [N, 4]

# soft label => hard label（以最大概率类别为预测）
preds_task1 = np.argmax(logits_task1, axis=-1)
preds_task2 = np.argmax(logits_task2, axis=-1)

# 验证集真实标签
labels_task1 = np.array([np.argmax(x) for x in val_ds["soft_label_task1"]])
labels_task2 = np.array([np.argmax(x) for x in val_ds["soft_label_task2"]])

# task1 分类结果
acc1 = accuracy_score(labels_task1, preds_task1)
prec1, rec1, f1_1, _ = precision_recall_fscore_support(labels_task1, preds_task1, average='macro')
print(f"Task1 宏平均F1={f1_1:.4f}, Acc={acc1:.4f}, Precision={prec1:.4f}, Recall={rec1:.4f}")

# task2 分类结果
acc2 = accuracy_score(labels_task2, preds_task2)
prec2, rec2, f1_2, _ = precision_recall_fscore_support(labels_task2, preds_task2, average='macro')
print(f"Task2 宏平均F1={f1_2:.4f}, Acc={acc2:.4f}, Precision={prec2:.4f}, Recall={rec2:.4f}")

# 详细报告
print("\nTask1 分类报告：")
print(classification_report(labels_task1, preds_task1, target_names=["NO", "YES"]))

print("\nTask2 分类报告：")
print(classification_report(labels_task2, preds_task2, target_names=["DIRECT", "REPORTED", "JUDGEMENTAL", "NO"]))

# 可选：混淆矩阵
print("Task1 混淆矩阵：\n", confusion_matrix(labels_task1, preds_task1))
print("Task2 混淆矩阵：\n", confusion_matrix(labels_task2, preds_task2))

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# task1
labels_task1 = np.array([np.argmax(x) for x in val_ds["soft_label_task1"]])
preds_task1 = np.argmax(eval_result.predictions[0], axis=-1)
cm1 = confusion_matrix(labels_task1, preds_task1)

plt.figure(figsize=(5,4))
sns.heatmap(cm1, annot=True, fmt='d', cmap="Blues",
            xticklabels=["NO", "YES"], yticklabels=["NO", "YES"])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Task1 Confusion Matrix')
plt.show()

# task2
labels_task2 = np.array([np.argmax(x) for x in val_ds["soft_label_task2"]])
preds_task2 = np.argmax(eval_result.predictions[1], axis=-1)
cm2 = confusion_matrix(labels_task2, preds_task2)

plt.figure(figsize=(7,5))
sns.heatmap(cm2, annot=True, fmt='d', cmap="Greens",
            xticklabels=["DIRECT", "REPORTED", "JUDGEMENTAL", "NO"],
            yticklabels=["DIRECT", "REPORTED", "JUDGEMENTAL", "NO"])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Task2 Confusion Matrix')
plt.show()

In [ ]:
# Task1
plt.figure(figsize=(4,3))
sns.countplot(x=labels_task1, palette="Blues")
plt.xticks([0,1], ["NO", "YES"])
plt.title("Task1 True Label Distribution")
plt.show()

# Task2
plt.figure(figsize=(6,3))
sns.countplot(x=labels_task2, palette="Greens")
plt.xticks([0,1,2,3], ["DIRECT", "REPORTED", "JUDGEMENTAL", "NO"])
plt.title("Task2 True Label Distribution")
plt.show()


In [ ]:
import pandas as pd

logs = pd.DataFrame(trainer.state.log_history)
plt.figure(figsize=(8,4))
plt.plot(logs[logs['loss'].notnull()]['step'], logs[logs['loss'].notnull()]['loss'], label='Train Loss')
plt.plot(logs[logs['eval_loss'].notnull()]['step'], logs[logs['eval_loss'].notnull()]['eval_loss'], label='Val Loss')
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 仅保留包含 eval 指标的记录
logs = logs.dropna(subset=["eval_task1_accuracy"])

plt.figure(figsize=(8,4))
plt.plot(
    logs["epoch"],                 # X 轴
    logs["eval_task1_accuracy"],   # Y 轴
    label="Task-1 Val Accuracy",
    linewidth=2                    # 线条粗细
)

plt.plot(
    logs["epoch"],
    logs["eval_task1_f1_macro"],
    label="Task-1 Val Macro-F1",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Task-1 validation curves")
plt.legend()
plt.grid(True, linestyle="--", alpha=.3)   # 细网格，可要可不要
plt.show()
logs = logs.dropna(subset=["eval_task2_accuracy"])

plt.figure(figsize=(8,4))
plt.plot(
    logs["epoch"],                 # X 轴
    logs["eval_task2_accuracy"],   # Y 轴
    label="Task-2 Val Accuracy",
    linewidth=2                    # 线条粗细
)

plt.plot(
    logs["epoch"],
    logs["eval_task2_f1_macro"],
    label="Task-2 Val Macro-F1",
    linewidth=2
)

plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Task-2 validation curves")
plt.legend()
plt.grid(True, linestyle="--", alpha=.3)   # 细网格，可要可不要
plt.show()

In [ ]:
from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# ① 推理得到验证集 logits
eval_out   = trainer.predict(val_ds)        # logits, labels=None
logits1, logits2 = eval_out.predictions     # 因为 forward 返回 (logits1, logits2)

# ② 把 soft-labels → hard label，得到 y_true / y_pred
y_true2 = np.argmax(val_ds["soft_label_task2"], axis=-1)
y_pred2 = np.argmax(logits2,                axis=-1)

# ③ sklearn 生成 report（输出为 dict）
report = classification_report(
    y_true2, y_pred2,
    target_names=["DIRECT","REPORTED","JUDGEMENTAL","NO"],
    output_dict=True
)

# ④ 把 dict → DataFrame，便于可视化/保存
rep_df = pd.DataFrame(report).T       # 每行：一个类别或总体 avg

# ------- 可选：保存到 csv --------
# rep_df.to_csv("task2_report.csv", index=True)

# ===============================================================
# 9. 画柱状图（纯线/无点也可以）
# ---------------------------------------------------------------
import seaborn as sns, matplotlib.pyplot as plt

class_names = ["DIRECT", "REPORTED", "JUDGEMENTAL", "NO"]
metrics     = ["precision", "recall", "f1-score"]

for m in metrics:
    plt.figure(figsize=(6,3))
    vals = [report[c][m] for c in class_names]               # 关键改动
    sns.barplot(x=class_names, y=vals, palette="viridis", legend=False)
    plt.ylim(0,1)
    plt.ylabel(m.capitalize());  plt.title(f"Task-2 {m.capitalize()} per class")
    plt.tight_layout();  plt.show()
# 查看report里所有key
print(report.keys())


In [ ]:
# ======================================================================
#  Task-3  ——  多标签评估 + 可视化
# ======================================================================
from sklearn.metrics import classification_report, multilabel_confusion_matrix
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt

# ------------------ 1. 准备 y_true / y_pred ---------------------------
LOGITS_T3 = eval_out.predictions[2]               # [N, 6]  ← 你的 forward 第3个输出
LABEL_NAMES_T3 = [
    "IDEOLOGICAL-INEQUALITY",
    "STEREOTYPING-DOMINANCE",
    "OBJECTIFICATION",
    "SEXUAL-VIOLENCE",
    "MISOGYNY-NON-SEXUAL-VIOLENCE",
    "NO"
]

# y_true: 验证集的 soft-label 已经是 multi-hot 向量 (N,6)
y_true3 = np.array(val_ds["soft_label_task3"]) > 0        # bool 矩阵

# y_pred: logits → sigmoid 概率 → 阈值化
prob3   = 1 / (1 + np.exp(-LOGITS_T3))                    # sigmoid
thr_vec = np.array([0.1666]*6)                            # 统一阈值 / 自己调
y_pred3 = prob3 >= thr_vec                                # bool 矩阵

# ------------------ 2. sklearn classification report ------------------
report3 = classification_report(
    y_true3, y_pred3,
    target_names=LABEL_NAMES_T3,
    output_dict=True,
    zero_division=0
)
rep3_df = pd.DataFrame(report3).T
print(rep3_df.round(3))

# ------------------ 3. per-label 混淆矩阵 ------------------------------
ml_cm = multilabel_confusion_matrix(y_true3, y_pred3)     # shape (6, 2, 2)

fig, axes = plt.subplots(2, 3, figsize=(13,7))
for idx, ax in enumerate(axes.flat):
    sns.heatmap(
        ml_cm[idx], annot=True, fmt='d', cmap="Oranges",
        xticklabels=["Pred 0","Pred 1"],
        yticklabels=["True 0","True 1"],
        ax=ax, cbar=False
    )
    ax.set_title(LABEL_NAMES_T3[idx], fontsize=9)
plt.suptitle("Task-3 per-label confusion matrices")
plt.tight_layout();  plt.show()

# ------------------ 4. 条形图：precision / recall / F1 -----------------
metrics = ["precision", "recall", "f1-score"]
for m in metrics:
    plt.figure(figsize=(8,3))
    vals = [report3[lab][m] for lab in LABEL_NAMES_T3]
    sns.barplot(x=LABEL_NAMES_T3, y=vals, palette="mako")
    plt.xticks(rotation=40, ha='right', fontsize=8)
    plt.ylim(0,1); plt.ylabel(m.capitalize())
    plt.title(f"Task-3 {m.capitalize()} per label")
    plt.tight_layout(); plt.show()

# ------------------ 5. 标签分布条形图 -----------------
plt.figure(figsize=(8,3))
pos_cnt = y_true3.sum(axis=0)           # 每列正样本数
sns.barplot(x=LABEL_NAMES_T3, y=pos_cnt, palette="rocket")
plt.xticks(rotation=40, ha='right', fontsize=8)
plt.ylabel("Positive samples")
plt.title("Task-3 true label distribution")
plt.tight_layout(); plt.show()